In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import shap
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA    = Path.cwd().parent / 'data' / 'interim'
FIGURES = Path.cwd().parent / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DAY      = 30
ONBOARDING_WINDOW = 90

eligible          = pd.read_csv(DATA / 'eligible_with_labels.csv')
new_clients       = pd.read_csv(DATA / 'new_clients.csv')
tx_with_reg       = pd.read_csv(DATA / 'tx_with_reg.csv')
messages          = pd.read_csv(DATA / 'messages.csv')
message_templates = pd.read_csv(DATA / 'message_templates.csv')
conversions       = pd.read_csv(DATA / 'conversions.csv')
business_units    = pd.read_csv(DATA / 'business_units.csv')

for df, col in [
    (new_clients, 'registration_date'),
    (tx_with_reg, 'date'),
    (tx_with_reg, 'registration_date'),
    (messages,    'send_date'),
    (conversions, 'date'),
]:
    df[col] = pd.to_datetime(df[col], utc=True)

# Compute days_since_first_tx: days relative to each client's first transaction
first_tx_reg_day = tx_with_reg.groupby('client_id')['days_since_reg'].min().rename('first_tx_reg_day')
tx_with_reg = tx_with_reg.join(first_tx_reg_day, on='client_id')
tx_with_reg['days_since_first_tx'] = tx_with_reg['days_since_reg'] - tx_with_reg['first_tx_reg_day']

print(f'eligible: {len(eligible):,}  SNAPSHOT_DAY={SNAPSHOT_DAY}')


In [ ]:
# Snapshot transactions (days 0..30 per client)
tx_snapshot = tx_with_reg[
    tx_with_reg['client_id'].isin(eligible['client_id']) &
    (tx_with_reg['days_since_first_tx'] >= 0) &
    (tx_with_reg['days_since_first_tx'] <= SNAPSHOT_DAY)
].copy()
tx_snapshot = tx_snapshot.merge(
    business_units[['id','location','category']], left_on='business_unit_id', right_on='id', how='left'
)

# First tx day per client (bridge for messages/conversions)
first_tx = (
    tx_with_reg.groupby('client_id')['days_since_reg'].min()
    .reset_index().rename(columns={'days_since_reg':'day_first_tx'})
)

# Format median gap from full eligible window (no leakage)
format_median_gap = (
    tx_with_reg[tx_with_reg['client_id'].isin(eligible['client_id'])]
    .merge(business_units[['id','location']], left_on='business_unit_id', right_on='id', how='left')
    .assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date'])
    .sort_values(['client_id','tx_date'])
    .assign(prev_day=lambda x: x.groupby('client_id')['tx_date'].shift(1))
    .assign(gap=lambda x: x['tx_date'] - x['prev_day'])
    .dropna(subset=['gap'])
    .groupby('location')['gap'].median()
    .reset_index().rename(columns={'gap':'median_gap','location':'first_location'})
)
global_median = format_median_gap['median_gap'].median()

print(f'Clients with tx in first {SNAPSHOT_DAY}d: {tx_snapshot["client_id"].nunique():,}')
print(f'Clients without:                         {eligible["client_id"].nunique() - tx_snapshot["client_id"].nunique():,}')


In [ ]:
# RFM + format-normalised features at day-30 snapshot
tx_count = (tx_snapshot.groupby('client_id')['transaction_id'].count()
            .reset_index().rename(columns={'transaction_id':'tx_count'}))
unique_days = (
    tx_snapshot.assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date'])
    .groupby('client_id')['tx_date'].count()
    .reset_index().rename(columns={'tx_date':'unique_active_days'})
)
recency = (tx_snapshot.groupby('client_id')['days_since_first_tx'].max()
           .reset_index().rename(columns={'days_since_first_tx':'last_tx_day'}))
recency['recency'] = SNAPSHOT_DAY - recency['last_tx_day']
recency = recency[['client_id','recency']]
monetary = (tx_snapshot.groupby('client_id').agg(
    total_spent=('payed_amount','sum'), avg_check=('payed_amount','mean'),
    bonuses_accum=('bonusses_accum','sum'), bonuses_used=('bonusses_used','sum')
).reset_index())
tx_sorted = (
    tx_snapshot.assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date']).sort_values(['client_id','tx_date'])
)
tx_sorted['prev_day'] = tx_sorted.groupby('client_id')['tx_date'].shift(1)
tx_sorted['gap']      = tx_sorted['tx_date'] - tx_sorted['prev_day']
avg_gap = (tx_sorted.groupby('client_id')['gap'].mean()
           .reset_index().rename(columns={'gap':'avg_days_between_tx'}))
avg_gap['avg_days_between_tx'] = avg_gap['avg_days_between_tx'].fillna(0)
half = SNAPSHOT_DAY // 2
first_half  = (tx_snapshot[tx_snapshot['days_since_first_tx'] <= half].groupby('client_id')['transaction_id'].count()
               .reset_index().rename(columns={'transaction_id':'tx_first_half'}))
second_half = (tx_snapshot[tx_snapshot['days_since_first_tx'] > half].groupby('client_id')['transaction_id'].count()
               .reset_index().rename(columns={'transaction_id':'tx_second_half'}))
trend = first_half.merge(second_half, on='client_id', how='outer').fillna(0)
trend['activity_trend'] = trend['tx_second_half'] - trend['tx_first_half']
trend = trend[['client_id','activity_trend']]
first_location = (
    tx_snapshot.sort_values('days_since_first_tx')
    .groupby('client_id')[['location','category']].first().reset_index()
    .rename(columns={'location':'first_location','category':'first_category'})
)
first_location = first_location.merge(format_median_gap, on='first_location', how='left')
first_location['median_gap'] = first_location['median_gap'].fillna(global_median)
unique_locations = (tx_snapshot.groupby('client_id')['location'].nunique()
                    .reset_index().rename(columns={'location':'unique_locations'}))
online_share = (tx_snapshot.groupby('client_id')
                .apply(lambda x: (x['category'] == 'Online purchases').mean())
                .reset_index().rename(columns={0:'online_tx_share'}))
recency_norm = recency.merge(first_location[['client_id','median_gap']], on='client_id', how='left')
recency_norm['recency_normalized'] = recency_norm['recency'] / recency_norm['median_gap']
recency_norm = recency_norm[['client_id','recency_normalized']]
visit_fulfill = unique_days.merge(first_location[['client_id','median_gap']], on='client_id', how='left')
visit_fulfill['expected_visits'] = SNAPSHOT_DAY / visit_fulfill['median_gap']
visit_fulfill['visit_fulfillment'] = visit_fulfill['unique_active_days'] / visit_fulfill['expected_visits']
visit_fulfill = visit_fulfill[['client_id','visit_fulfillment']]
print('RFM + format-normalised features done')


In [ ]:
# Communication and conversion features (snapshot window only)
segment_groups = {
    'Thank-you / post-visit':'auto_trigger','Quest / cross-venue':'auto_trigger',
    'Loyalty: double points':'retention_offer','Loyalty: points spend/expiry':'retention_offer',
    'Coupon / discount':'retention_offer',
}
message_templates['segment_group'] = message_templates['segment'].map(segment_groups).fillna('other')
bridge = new_clients[['client_id','loyalty_client_id']].copy()

messages_new = (
    messages.dropna(subset=['client_id'])
    .merge(bridge, left_on='client_id', right_on='loyalty_client_id', how='inner')
    .rename(columns={'client_id_x':'loyalty_client_id_msg','client_id_y':'client_id'})
    .merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
    .merge(first_tx, on='client_id', how='left')
)
messages_new['days_since_first_tx'] = (
    (messages_new['send_date'] - messages_new['registration_date']).dt.total_seconds() / 86400
    - messages_new['day_first_tx']
)
msg_snapshot = messages_new[
    (messages_new['days_since_first_tx'] >= 0) &
    (messages_new['days_since_first_tx'] <= SNAPSHOT_DAY) &
    (messages_new['client_id'].isin(eligible['client_id']))
].merge(message_templates[['message_template_id','segment_group']], on='message_template_id', how='left')
msg_features = (
    msg_snapshot.groupby('client_id')
    .agg(_n=('message_id','count'), _op=('status', lambda x: (x=='Opened').sum()))
    .reset_index()
)
msg_features['open_rate'] = msg_features['_op'] / msg_features['_n']
msg_features = msg_features[['client_id','open_rate']]

conv_new = (
    conversions
    .merge(bridge, left_on='client_id', right_on='loyalty_client_id', how='inner')
    .rename(columns={'client_id_x':'loyalty_client_id_conv','client_id_y':'client_id'})
    .merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
    .merge(first_tx, on='client_id', how='left')
)
conv_new['days_since_first_tx'] = (
    (conv_new['date'] - conv_new['registration_date']).dt.total_seconds() / 86400
    - conv_new['day_first_tx']
)
conv_snapshot = conv_new[
    (conv_new['days_since_first_tx'] >= 0) &
    (conv_new['days_since_first_tx'] <= SNAPSHOT_DAY) &
    (conv_new['client_id'].isin(eligible['client_id']))
]
conv_features = (
    conv_snapshot.groupby('client_id').agg(conv_count=('conversion_id','count')).reset_index()
)

profile = new_clients[new_clients['client_id'].isin(eligible['client_id'])][['client_id','registration_date']].copy()
profile['registration_date'] = pd.to_datetime(profile['registration_date']).dt.tz_localize(None)
profile['reg_day_of_week'] = profile['registration_date'].dt.dayofweek
profile['reg_month']       = profile['registration_date'].dt.month
profile = profile[['client_id','reg_day_of_week','reg_month']]
print('Communication and conversion features done')


In [ ]:
# Assemble 30-day feature matrix
features30 = eligible[['client_id','churned']].copy()
for df in [tx_count, unique_days, recency, monetary, avg_gap, trend,
           first_location[['client_id','first_location','first_category']],
           unique_locations, online_share, recency_norm, visit_fulfill]:
    features30 = features30.merge(df, on='client_id', how='left')
features30 = features30.merge(msg_features, on='client_id', how='left')
features30 = features30.merge(conv_features, on='client_id', how='left')
features30 = features30.merge(profile, on='client_id', how='left')

for col in ['avg_check','total_spent','tx_count','unique_active_days','activity_trend',
            'open_rate','conv_count','avg_days_between_tx','bonuses_accum','bonuses_used',
            'unique_locations','online_tx_share']:
    features30[col] = features30[col].fillna(0)
features30['recency']            = features30['recency'].fillna(SNAPSHOT_DAY)
features30['recency_normalized'] = features30['recency_normalized'].fillna(SNAPSHOT_DAY / global_median)
features30['visit_fulfillment']  = features30['visit_fulfillment'].fillna(0)
features30['first_location']     = features30['first_location'].fillna('Unknown')
features30['first_category']     = features30['first_category'].fillna('Unknown')

print(f'features30: {features30.shape}  churn: {features30["churned"].mean()*100:.1f}%')
print(f'NaN remaining: {features30.isna().sum().sum()}')


In [ ]:
# Temporal split
features30 = features30.merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
features30['registration_date'] = pd.to_datetime(features30['registration_date'], utc=True)

train_mask = features30['registration_date'] < pd.Timestamp('2031-06-01', tz='UTC')
val_mask   = (features30['registration_date'] >= pd.Timestamp('2031-06-01', tz='UTC')) & \
             (features30['registration_date'] <  pd.Timestamp('2031-08-01', tz='UTC'))
test_mask  = features30['registration_date'] >= pd.Timestamp('2031-08-01', tz='UTC')

print(f'Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  Test: {test_mask.sum():,}')


In [ ]:
# Train LR, RF, XGBoost on 30-day features
DROP_COLS = ['client_id','churned','registration_date']
CAT_COLS  = ['first_location','first_category']
df_train = features30[train_mask].copy()
df_val   = features30[val_mask].copy()
df_test  = features30[test_mask].copy()
le_dict = {}
for col in CAT_COLS:
    le = LabelEncoder(); le.fit(df_train[col])
    df_train[col] = le.transform(df_train[col])
    df_val[col]   = df_val[col].map(dict(zip(le.classes_, le.transform(le.classes_)))).fillna(-1).astype(int)
    df_test[col]  = df_test[col].map(dict(zip(le.classes_, le.transform(le.classes_)))).fillna(-1).astype(int)
    le_dict[col]  = le
X_train30 = df_train.drop(columns=DROP_COLS); y_train30 = df_train['churned']
X_val30   = df_val.drop(columns=DROP_COLS);   y_val30   = df_val['churned']
X_test30  = df_test.drop(columns=DROP_COLS);  y_test30  = df_test['churned']

lr30 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr30.fit(X_train30, y_train30)
lr30_test_proba = lr30.predict_proba(X_test30)[:,1]

rf30 = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf30.fit(X_train30, y_train30)
rf30_val_auc    = roc_auc_score(y_val30, rf30.predict_proba(X_val30)[:,1])
rf30_test_proba = rf30.predict_proba(X_test30)[:,1]
rf30_test_auc   = roc_auc_score(y_test30, rf30_test_proba)

scale_pw = (y_train30 == 0).sum() / (y_train30 == 1).sum()
xgb30 = xgb.XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pw,
    random_state=42, eval_metric='auc', early_stopping_rounds=50, verbosity=0)
xgb30.fit(X_train30, y_train30, eval_set=[(X_val30, y_val30)], verbose=False)
xgb30_test_proba = xgb30.predict_proba(X_test30)[:,1]
xgb30_test_auc   = roc_auc_score(y_test30, xgb30_test_proba)

# Compare 90-day vs 30-day
preds_90 = pd.read_csv(DATA / 'test_predictions.csv')
auc_90_rf = roc_auc_score(preds_90['churned'], preds_90['proba_rf'])

print(f"{'Model':<22} {'90-day':>9} {'30-day':>9} {'Delta':>8}")
print('-' * 50)
print(f"  {'Random Forest':<20} {auc_90_rf:>9.4f} {rf30_test_auc:>9.4f} {rf30_test_auc-auc_90_rf:>+8.4f}")
print(f"\n2-feature baseline AUC: ", end='')
rf30_simple = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf30_simple.fit(X_train30[['recency','tx_count']], y_train30)
simple_auc30 = roc_auc_score(y_test30, rf30_simple.predict_proba(X_test30[['recency','tx_count']])[:,1])
print(f'{simple_auc30:.4f}  full: {rf30_test_auc:.4f}  gain: {rf30_test_auc-simple_auc30:+.4f}')


In [ ]:
# Save 30-day model and predictions
with open(DATA / 'best_model_rf30.pkl', 'wb') as f:
    pickle.dump(rf30, f)
X_test30.to_csv(DATA / 'X_test30.csv', index=False)
y_test30.to_csv(DATA / 'y_test30.csv', index=False)
pd.DataFrame({
    'client_id': df_test['client_id'].values,
    'churned':   y_test30.values,
    'proba_rf30':  rf30_test_proba,
    'proba_xgb30': xgb30_test_proba,
    'proba_lr30':  lr30.predict_proba(X_test30)[:,1],
}).to_csv(DATA / 'test_predictions_30day.csv', index=False)
print('Saved: best_model_rf30.pkl, X/y_test30.csv, test_predictions_30day.csv')
